# mutation_burden.ipynb
- Do ecDNA+ tumors have greater somatic mutation burden?
- Do ecDNA+ tumors have greater likely pathogenic somatic mutation burden?*
- Do ecDNA+ tumors have greater pathogenic somatic mutation burden?**

\* Impact HIGH or MODERATE, Polyphen + SIFT deleterious if available, and within a cancer gene.  
** (PBTA only) pathogenicity from HotSpot annotations.***  
*** Hotspots described in  
    https://github.com/kids-first/kf-annotation-tools/blob/master/docs/SOMATIC_SNV_ANNOT_README.md#source-specific-inputs  
    https://www.cancerhotspots.org/#/about
    

We compared variant count and stratified ecDNA by tumor type.

### RESULTS  
Mann-Whitney U test p-values, all tumors:
|         |  all |    lp|     p|
|---------|------|------|------|
|ec vs na |2e-14*|2e-11*| 9e-5*|
|ec vs chr|  0.23| 0.02*|  0.10|
|chr vs na|2e-18*|6e-18*|5e-11*|

MWU test p-values, MBL tumors:
|         |  all |   lp|    p|
|---------|------|-----|-----|
|ec vs na |0.002*| 0.22| 0.14|
|ec vs chr|  0.79| 0.06| 0.17|
|chr vs na|  0.16| 0.16| 0.57|

MWU test p-values, HGG tumors:
|         |  all |    lp|    p|
|---------|------|------|-----|
|ec vs na |0.004*|  0.15| 0.48|
|ec vs chr|  0.59|  0.33| 0.33|
|chr vs na|0.007*|0.003*| 0.68|

## Data import
Parse SNV data into a DataFrame indexed by biosample with columns: 
| biosample_id | cancer_type | amplicon_class | all_counts | likely_pathogenic_counts | pathogenic_counts |
|--------------|-------------|----------------|------------|--------------------------|-------------------|
| `str`        | `str`       | `str`          | `int`      | `int`                    | `dbl`             | 

In [ ]:
import pandas as pd
import os
from pathlib import Path

OUT_DIR=Path('out')
OUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
def parse_bcftools_stats(file):
    # read a bcftools stats output, and
    # return the total number of variants
    try:
        with open(file) as f:
            for line in f:
                if line.startswith('SN	0	number of records:'):
                    count=line.rstrip().split('\t')[-1]
                    count=int(count)
        return count
    except:
        print("could not read file:",file)
        raise

In [ ]:
def get_variant_counts(path):
    # for all .stats.txt files in path,
    # create a dictionary of the format:
    # return {sample : count}
    d = {}
    for filename in os.listdir(path):
        full_path = os.path.join(path, filename)
        if full_path.endswith('.stats.txt'):
            count = parse_bcftools_stats(full_path)
            #print(count)
            filename = filename.split('.')[0]
            d[filename] = count
    return d

def get_variant_count_df():
    # return a dataframe of the format:
    # file_id, all_counts, likely_pathogenic_counts, pathogenic_counts

    columns = ['all','likely_pathogenic','pathogenic']
    for i in range(len(columns)):
        col = columns[i]
        counts = get_variant_counts(f'./data/stats/{col}')
        s = pd.Series(counts)
        s.name = f"{col}_counts"
        if i == 0:
            df = pd.DataFrame(s)
        else:
            df = df.join(s)
    return df

In [ ]:
get_variant_count_df().tail()

In [ ]:
def read_cbtn_manifest(file='../data/variants/metadata/manifest_20250910_143948.csv'):
    df = pd.read_csv(file)
    df=df[df.name.str.endswith('.gz')]
    df['index'] = df.name.map(lambda x: os.path.basename(x).split('.')[0])
    df = df.set_index('index')
    df = df.rename(columns={'Kids First Biospecimen ID':'biosample_id'})
    df = df[['biosample_id']].copy()
    return df
def read_sj_manifest(file='../data/variants/metadata/sj_somatic_vcfs.tsv'):
    df = pd.read_table(file)
    df = df[df['vcf_file_name'] != 'SJST030131_D3_G1.Somatic.vcf.gz'] # this file is corrupted
    df['index'] = df["vcf_file_name"].str.rsplit(".", n=-1).str[0]
    df = df.set_index('index')
    df = df.rename(columns={'sample_name':'biosample_id'})
    return df[['biosample_id']].copy()
def read_manifest():
    d1 = read_cbtn_manifest()
    d2 = read_sj_manifest()
    return pd.concat([d1,d2])

In [ ]:
def read_biosample_table(file='../data/Supplementary Tables.xlsx'):
    df = pd.read_excel(file,sheet_name='2. Biosamples',index_col='biosample_id')
    df = df.drop(columns=['external_sample_id','file_name','in_unique_tumor_set','in_unique_patient_set'])
    df["amplicon_class"] = df["amplicon_class"].replace({
        "intrachromosomal": "chromosomal",
    })
    return df.copy()

def merge_counts(manifest,counts):
    df = manifest.join(counts,how='inner')
    return df

def check_file_bs_ids(example):
    # check no duplicates
    assert len(example.index) == len(example.index.unique())
    assert len(example['biosample_id']) == len(example['biosample_id'].unique())
    # check no missing values
    assert sum(example.index.isna()) == 0
    assert sum(example['biosample_id'].isna()) == 0
    # check 1:1 mapping
    assert len(example.index) == len(example['biosample_id'])
    return

def merge_ecDNA_counts(ecDNA_df,counts_df):
    check_file_bs_ids(counts_df)
    counts_df['file_id'] = counts_df.index
    counts_df = counts_df.set_index('biosample_id')
    df = ecDNA_df.join(counts_df,how='inner')
    df = df.sort_values(by=['patient_id','amplicon_class','ecDNA_sequences_detected','age_at_diagnosis'],
                   ascending=[True,False,True,True])
    df["in_snv_set"]=~df.duplicated(subset=["patient_id"],keep='last')
    #df = df[df.in_snv_set].copy()
    return df

In [ ]:
df = merge_ecDNA_counts(read_biosample_table(),
                   merge_counts(read_manifest(),get_variant_count_df())
                  )
# write df, including duplicates, then drop duplicates
path=OUT_DIR/'biosample_mutation_burden.tsv'
df.to_csv(path,sep='\t')

df = df[df.in_snv_set]
df

In [ ]:
df['amplicon_class'].value_counts()

# Tests and plots
Compare mutation burden across amplicon classes by Wilcoxon rank sum test,
make boxplots

In [ ]:
import scipy
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def compare_somatic_mutation_burden(df, burden_colname, log=True, test=scipy.stats.mannwhitneyu):
    # parameter parsing
    if log:
        df['y_counts'] = np.log1p(df[burden_colname])
        ylab = 'Log (mutation burden)'
    else:
        df['y_counts'] = df[burden_colname]
        ylab = 'mutation burden'
    df = df.dropna(axis=0,subset=['y_counts'])
    
    # plot
    ax = sns.violinplot(
        data = df,
        x = 'amplicon_class',
        y = 'y_counts',
        order = ['ecDNA','chromosomal','no amplification',]
    )

    # figure post-formatting
    sns.despine()
    ax.set_ylabel(ylab)
    ax.set_xlabel(None)
    ax.set_ylim(bottom=0)
    cts = df["amplicon_class"].value_counts().to_dict()
    new_labels = [
        f"{label.get_text()}\n(n={cts[label.get_text()]})"
        for label in ax.get_xticklabels()
    ]
    ax.set_xticklabels(new_labels)

    # statistics
    classes = list(df.amplicon_class.unique()) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['y_counts']
            b = df[df.amplicon_class == classes[j]]['y_counts']
            print(test(a,b))
    return ax

In [ ]:
ax = compare_somatic_mutation_burden(df,'all_counts')

plt.savefig('out/mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/mutation_burden_p.svg')
#plt.show()

In [ ]:
def stratify_ecDNA_by_tumor_type(df):
    df = df.loc[:,['amplicon_class','cancer_type']]
    df = df.groupby(['cancer_type','amplicon_class']).size()
    return df

## Stratifications by tumor type

In [ ]:
strat = stratify_ecDNA_by_tumor_type(df)
strat

In [ ]:
df_MBL = df[df['cancer_type'].isin(['MBL'])].copy()
df_HGG = df[df['cancer_type'].isin(['HGG'])].copy()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'all_counts')
plt.savefig('out/MBL_mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/MBL_mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/MBL_mutation_burden_p.svg')
#plt.show()

In [ ]:
compare_somatic_mutation_burden(df_HGG,'all_counts')
plt.savefig('out/HGG_mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_HGG,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/HGG_mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_HGG,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/HGG_mutation_burden_p.svg')
#plt.show()